In [1]:
#Imports ?
import numpy as np
import pandas as pd
import xarray as xr

from netCDF4 import Dataset

import matplotlib.pyplot as plt
#from mpl_toolkits.basemap import Basemap

import cartopy.crs as ccrs
import cartopy.feature as cfeature

from scipy import stats
from scipy import ndimage
from scipy.stats import rankdata

**Calculate seasonal anomalies of precip**

In [2]:
precip= xr.open_dataset('../../Data-Other/Article3_data/CDR.nc', engine='netcdf4') 
precip

<xarray.Dataset> Size: 10MB
Dimensions:   (datetime: 487, lat: 68, lon: 73)
Coordinates:
  * datetime  (datetime) datetime64[ns] 4kB 1983-01-01 1983-02-01 ... 2023-07-01
  * lat       (lat) float32 272B 37.0 36.75 36.5 36.25 ... 21.0 20.75 20.5 20.25
  * lon       (lon) float32 292B -18.0 -17.75 -17.5 -17.25 ... -0.5 -0.25 0.0
Data variables:
    crs       int32 4B ...
    precip    (datetime, lat, lon) float32 10MB ...
Attributes:
    Conventions:  CF-1.6

In [ ]:
#Code ?
# 1. Create date coordinate efficiently without making a copy of the dataset 
cru.coords['date'] = pd.to_datetime(cru['time'].values.astype(str)) 
# 2. Extract relevant variables directly to reduce memory load 
df = cru['pre'].to_dataframe().dropna().reset_index() 
# # 3. Extract year and month efficiently 
df['year'] = df['time'].dt.year 
df['month'] = df['time'].dt.month 
# **FAST NumPy Vectorized Method for Assigning Seasons** 
seasons = np.array(["DJF", "MAM", "JJA", "SON"]) 
# Define conditions for season assignment 
winter = df['month'].isin([12, 1, 2]) 
spring = df['month'].isin([3, 4, 5]) 
summer = df['month'].isin([6, 7, 8]) 
autumn = df['month'].isin([9, 10, 11]) 
# Assign seasons using np.select (much faster than apply) 
df['season'] = np.select([winter, spring, summer, autumn], seasons, default=None) 
# 5. Adjust season_year directly 
df['season_year'] = df['year'].values 
df.loc[df['month'] == 12, 'season_year'] += 1

In [ ]:
precip.index = pd.to_datetime(precip.index) 
df = precip.copy() 
df['year'] = df.index.year 
df['month'] = df.index.month 
# Subset only reference period 
climatology_base = df[(df['year'] >= 1991) & (df['year'] <= 2020)] 
# Monthly climatological mean for each grid point 
monthly_clim_mean = climatology_base.groupby('month').mean() 
#Calculate anomalies 
df_anom = df.copy() 
df_anom.drop(columns=['year'], inplace=True) 
# For each month, apply (P - clim) / clim * 100 
for m in range(1, 13): 
    clim_vals = monthly_clim_mean.loc[m] 
    df_anom.loc[df_anom['month'] == m, df_anom.columns.difference(['month'])] = ( (df_anom.loc[df_anom['month'] == m, df_anom.columns.difference(['month'])] - clim_vals) / clim_vals ) * 100 
# Assign seasons 
seasons = np.array(["DJF", "MAM", "JJA", "SON"]) 
winter = df_anom['month'].isin([12, 1, 2]) 
spring = df_anom['month'].isin([3, 4, 5]) 
summer = df_anom['month'].isin([6, 7, 8]) 
autumn = df_anom['month'].isin([9, 10, 11]) 
df_anom['season'] = np.select([winter, spring, summer, autumn], seasons, default=None) 
df_anom['season_year'] = df_anom.index.year 
df_anom.loc[df_anom['month'] == 12, 'season_year'] += 1 
df_anom.drop(columns=['month'], inplace=True) 
# Seasonal 
seasonal_anom = df_anom.groupby(['season_year', 'season']).mean().sort_index() 
# Annual 
df_anom['year'] = df_anom.index.year 
annual_anom = df_anom.groupby('year').mean() 
seasonal_anom

**Calculate annual anomalies of precip**

In [ ]:
#Code ?

**Plotting the time-series of precipitation anomalies (annual and seasonal)**

In [ ]:
import matplotlib.pyplot as plt 
import numpy as np 
from scipy.stats import linregress 

# Subset the required seasons 
son_df = seasonal_anom.xs('SON', level='season') 
djf_df = seasonal_anom.xs('DJF', level='season') 
mam_df = seasonal_anom.xs('MAM', level='season') 
# Function to calculate summary stats for plotting
def compute_stats(df):
    df_stats = pd.DataFrame({
        'mean': df.mean(axis=1),
        'p5': df.quantile(0.05, axis=1),
        'p95': df.quantile(0.95, axis=1),
        'min': df.min(axis=1),
        'max': df.max(axis=1)
    })
    df_stats.index.name = 'season_year'
    df_stats.reset_index(inplace=True)
    return df_stats
# Compute stats for each season
ann_stats = compute_stats(annual_anom)
son_stats = compute_stats(son_df)
djf_stats = compute_stats(djf_df)
mam_stats = compute_stats(mam_df)
# List of all stats and labels
panels = [('ANN', ann_stats), ('SON', son_stats), ('DJF', djf_stats), ('MAM', mam_stats)]
# === PLOT ===
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharey=True)
axes = axes.flatten()
for i, (title, stats) in enumerate(panels):
    ax = axes[i]
    # Plot mean line
    ax.plot(stats['season_year'], stats['mean'], color='black', label='Mean SPEI')
    ax.fill_between(stats['season_year'], stats['p5'], stats['p95'], color='blue', alpha=0.3, label='5th–95th Percentile')
    ax.plot(stats['season_year'], stats['min'], linestyle='--', color='black', label='Min–Max Boundaries' if i == 0 else "")
    ax.plot(stats['season_year'], stats['max'], linestyle='--', color='black')
    # Trend line calculation
    x = stats['season_year'].values
    y = stats['mean'].values
    slope, intercept, *_ = linregress(x, y)
    trend_per_decade = slope * 10
    ax.text(x.max() - 8, 81, f"Trend = {trend_per_decade:.3f}mm/decade", fontsize=10, color='black')
    # X-axis ticks: marks every year, label every 3 years
    years = np.arange(x.min(), x.max() + 1)
    year_labels = [str(y) if y % 3 == 0 else "" for y in years]
    ax.set_xticks(years)
    ax.set_xticklabels(year_labels, rotation=0)
    # Highlight 5 driest years
    lowest = stats.nsmallest(5, 'mean')
    ax.scatter(lowest['season_year'], lowest['mean'], color='red', zorder=5)
    for _, row in lowest.iterrows():
        ax.text(row['season_year'], row['mean'] - 18, str(int(row['season_year'])), color='red', fontsize=9, ha='center', rotation=90)
    # Highlight anomalies >= 80mm
    clip_val = 80  # visible y-axis threshold
    # Identify years where the true max exceeds the visual cap
    outliers = stats[stats['max'] > clip_val]
    # Annotate the actual value just above the triangle
    for _, row in outliers.iterrows():
        ax.text(row['season_year'], clip_val + 4, f"{row['max']:.0f}", color='black', fontsize=8, ha='center', va='bottom', rotation=90)
    ax.set_title(title, fontdict={'weight':'bold'})
    if i % 2 == 0:
        ax.set_ylabel("Precipitation Anomaly (mm)")
    ax.grid(axis='y')
    # Visually cap the y-axis range
    ax.set_ylim(-60, 80)
# Add legend only once
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.subplots_adjust(bottom=0.05)
plt.show()

**SPEI12 plots**

In [ ]:
import pandas as pd
import numpy as np

# Ensure datetime index
spei_df.index = pd.to_datetime(spei_df.index)

# Step 1: Extract year and month
df = spei_df.copy()
df['year'] = df.index.year
df['month'] = df.index.month

# Step 2: Assign seasons
seasons = np.array(["DJF", "MAM", "JJA", "SON"])
winter = df['month'].isin([12, 1, 2])
spring = df['month'].isin([3, 4, 5])
summer = df['month'].isin([6, 7, 8])
autumn = df['month'].isin([9, 10, 11])

df['season'] = np.select([winter, spring, summer, autumn], seasons, default=None)

# Step 3: Adjust season_year for DJF (shift December to next year)
df['season_year'] = df['year']
df.loc[df['month'] == 12, 'season_year'] += 1

# Step 4: Drop month/year for aggregation
df.drop(columns=['month', 'year'], inplace=True)

# Step 5: Group by season and season_year and compute seasonal mean
seasonal_mean = df.groupby(['season_year', 'season']).mean().sort_index()

In [ ]:
# Replace -inf and inf with NaN
spei_clean = spei_df.replace([np.inf, -np.inf], np.nan)
spei_clean['year'] = spei_clean.index.year
annual_mean = spei_clean.groupby('year').mean()

# Compute summary stats
ann_stats = pd.DataFrame({
    'season_year': annual_mean.index,
    'mean': annual_mean.mean(axis=1),
    'p5': annual_mean.quantile(0.05, axis=1),
    'p95': annual_mean.quantile(0.95, axis=1),
    'min': annual_mean.min(axis=1),
    'max': annual_mean.max(axis=1)
})

In [ ]:
import matplotlib.pyplot as plt 
import numpy as np 
from scipy.stats import linregress 

# Subset the required seasons 
son_df = seasonal_mean.xs('SON', level='season') 
djf_df = seasonal_mean.xs('DJF', level='season') 
mam_df = seasonal_mean.xs('MAM', level='season') 
# Function to calculate summary stats for plotting
def compute_stats(df):
    df_stats = pd.DataFrame({
        'mean': df.mean(axis=1),
        'p5': df.quantile(0.05, axis=1),
        'p95': df.quantile(0.95, axis=1),
        'min': df.min(axis=1),
        'max': df.max(axis=1)
    })
    df_stats.index.name = 'season_year'
    df_stats.reset_index(inplace=True)
    return df_stats

# Compute stats for each season
ann_stats = compute_stats(annual_mean)
son_stats = compute_stats(son_df)
djf_stats = compute_stats(djf_df)
mam_stats = compute_stats(mam_df)

# List of all stats and labels
panels = [('ANN', ann_stats), ('SON', son_stats), ('DJF', djf_stats), ('MAM', mam_stats)]

# === PLOT ===
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
fig, axes = plt.subplots(2, 2, figsize=(16, 8), sharey=True)
axes = axes.flatten()

for i, (title, stats) in enumerate(panels):
    ax = axes[i]
    # Plot mean line
    ax.plot(stats['season_year'], stats['mean'], color='black', label='Mean SPEI')
    ax.fill_between(stats['season_year'], stats['p5'], stats['p95'], color='orange', alpha=0.3, label='5th–95th Percentile')
    ax.plot(stats['season_year'], stats['min'], linestyle='--', color='black', label='Min–Max Boundaries' if i == 0 else "")
    ax.plot(stats['season_year'], stats['max'], linestyle='--', color='black')

     # === Trend line calculation ===
    x = stats['season_year'].values
    y = stats['mean'].values
    slope, intercept, r_value, p_value, std_err = linregress(x, y)
    trend_per_decade = slope * 10

    # Plot the trend line
    ax.plot(x, intercept + slope * x, color='blue', linestyle='-', linewidth=1.2, label='Trend line')

    # Add text for trend and p-value
    ax.text(
        x.min(), -2.7,
        f"Trend = {trend_per_decade:.3f}/decade\np = {p_value:.3f}",
        fontsize=9,
        color='blue',
        va='top'
    )

    # X-axis ticks: marks every year, label every 3 years
    years = np.arange(x.min(), x.max() + 1)
    year_labels = [str(y) if y % 3 == 0 else "" for y in years]
    ax.set_xticks(years)
    ax.set_xticklabels(year_labels, rotation=0)

    # Highlight 5 driest years
    lowest = stats.nsmallest(5, 'mean')
    ax.scatter(lowest['season_year'], lowest['mean'], color='red', zorder=5)
    for _, row in lowest.iterrows():
        ax.text(row['season_year'], row['mean'] - 0.6, str(int(row['season_year'])), color='red', fontsize=9, ha='center', rotation=90)

    ax.set_title(title, fontdict={'weight':'bold'})
    if i % 2 == 0:
        ax.set_ylabel("SPEI12", fontdict={'weight':'bold'})
    ax.grid(axis='y')

# Add legend only once
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=3, bbox_to_anchor=(0.5, -0.05))

plt.tight_layout()
plt.subplots_adjust(bottom=0.05)
plt.show()